# A Deterministic Merkle Radix dictionary, actually built

The proposal under test: a triple-store **dictionary** (URI/literal → integer id) stored as a
prefix-compressed radix trie whose nodes are content-addressed — "a Deterministic Merkle Radix
Prolly Tree" — claiming prefix deduplication on repetitive namespaces, history independence, and
address-pruned diffs. This notebook is the proof-of-work: a real implementation
([`merkle_radix_dict.py`](merkle_radix_dict.py), property-tested by
`test_merkle_radix_dict.py`), measured on an NCIt-shaped corpus.

Two corrections to the design sketch this notebook set out to prove, established before any
measurement:

1. **History independence needs no chunker.** The compressed radix trie of a key set is
   *unique* — its shape is a function of the keys alone. Canonical serialization (edges sorted
   by byte) is the only extra requirement. The sketch's "Gaussian CDF split rule" is
   unnecessary for the property it was invoked for. The property is pinned the strong way in
   the test suite: folding path-copying inserts in hypothesis-random orders produces a root
   **byte-identical** to the canonical batch build, for arbitrary key sets.
2. **Where content-defined chunking genuinely enters** is storage packing: radix nodes are many
   and small, pages want ~4 KiB — so the canonical node sequence is packed into pages with a
   boundary rule on node addresses (§5). That is the actual "prolly" marriage.

(The sketch's own Go proof-of-concept does not demonstrate its claims: its test data contains
duplicate keys whose ids clobber each other in a first-character edge map, and its
`fmt.Sprintf("http://xmlns.com", workerID, j)` has no format verbs — every worker inserts the
same string, so the matching-roots "SUCCESS" is vacuous. Hence this notebook.)

## 1. An NCIt-shaped corpus

The National Cancer Institute Thesaurus is the stress case the design targets: hundreds of
thousands of terms under a *single* long namespace
(`http://purl.obolibrary.org/obo/NCIT_C…`), plus smaller vocabularies. We generate that shape —
40k NCIT concepts, 2k GO terms, the usual RDF/OWL/SKOS property URIs, and 8k label literals:

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import merkle_radix_dict as mrd

plt.rcParams.update({"figure.figsize": (7.5, 3.0), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})

def ncit_corpus():
    entries = {}
    nid = 0
    def add(k):
        nonlocal nid
        entries[k.encode()] = nid
        nid += 1
    for i in range(40_000):
        add(f"http://purl.obolibrary.org/obo/NCIT_C{100000 + i}")
    for i in range(2_000):
        add(f"http://purl.obolibrary.org/obo/GO_{7000000 + i}")
    for name in ("label", "comment", "subClassOf", "seeAlso", "isDefinedBy"):
        add(f"http://www.w3.org/2000/01/rdf-schema#{name}")
    for name in ("Class", "ObjectProperty", "DatatypeProperty", "equivalentClass"):
        add(f"http://www.w3.org/2002/07/owl#{name}")
    for i in range(8_000):
        add(f"Neoplasm of anatomic site {i} (morphology variant)")
    return entries

ENTRIES = ncit_corpus()
raw_bytes = sum(len(k) for k in ENTRIES)
print(f"{len(ENTRIES):,} dictionary entries, {raw_bytes/1e6:.1f} MB of raw key bytes")

50,009 dictionary entries, 2.2 MB of raw key bytes


## 2. History independence, demonstrated on the real corpus

Build the whole dictionary canonically; then fold single inserts over three shuffled orders of
all 50k entries. Byte-identical roots each time (the test suite proves this ∀ key sets via
hypothesis; this is the same property at corpus scale):

In [2]:
import random, time

t0 = time.perf_counter()
store = mrd.Store()
root = mrd.build(store, ENTRIES)
print(f"canonical build: {time.perf_counter() - t0:.1f}s, root {root.hex()}")

for trial in range(3):
    items = list(ENTRIES.items())
    random.Random(trial).shuffle(items)
    s2 = mrd.Store()
    r2 = mrd.build(s2, {items[0][0]: items[0][1]})
    for k, v in items[1:]:
        r2 = mrd.insert(s2, r2, k, v)
    assert r2 == root
print("3 shuffled insert-folds of 50k entries → byte-identical root  ✓")

canonical build: 0.3s, root d8eedb219a954a8b2da6bbda245725583fa1ebaf


3 shuffled insert-folds of 50k entries → byte-identical root  ✓


## 3. Prefix deduplication — the headline claim, measured honestly

Compare three storages of the same dictionary: raw key bytes; the radix structure (every shared
prefix stored once — plus the structural overhead of addresses and edges); and both after
`zlib`, because the honest comparison includes what block compression would do to a *flat
sorted* layout (where shared prefixes sit adjacent and compress superbly):

In [3]:
import zlib

structural = mrd.stored_bytes(store, root)
flat_sorted = b"".join(sorted(ENTRIES))                      # flat layout: full strings, sorted
node_blob = b"".join(store.get(a).serialize() for a in mrd.canonical_order(store, root))

print(f"{'layout':>28} {'bytes':>12} {'vs raw':>8}")
print(f"{'raw unique strings':>28} {raw_bytes:>12,} {'100%':>8}")
print(f"{'radix structural':>28} {structural:>12,} {structural/raw_bytes:>7.0%}")
print(f"{'flat sorted + zlib':>28} {len(zlib.compress(flat_sorted, 6)):>12,} "
      f"{len(zlib.compress(flat_sorted, 6))/raw_bytes:>7.0%}")
print(f"{'radix nodes + zlib':>28} {len(zlib.compress(node_blob, 6)):>12,} "
      f"{len(zlib.compress(node_blob, 6))/raw_bytes:>7.0%}")

                      layout        bytes   vs raw
          raw unique strings    2,209,285     100%
            radix structural    2,026,441     92%
          flat sorted + zlib      129,056      6%


          radix nodes + zlib    1,348,348     61%


The measurement is harsher than the sketch — and than this notebook's own first draft. The
radix structure saves prefix bytes but **pays for them in Merkle addresses**: ~55k edges × 20
bytes of hash is ~1.1 MB of incompressible pointer material, so the structural footprint lands
at ~92% of raw — a wash, not a reduction. And a flat **sorted** layout plus zlib crushes both
(shared prefixes sit adjacent; the compressor eats them), while the radix blob resists
compression precisely because hashes are incompressible.

So the honest accounting: **prefix deduplication does not pay for this structure on footprint —
the operational properties do**: looking up a key without decompressing the dictionary (§4's
bounded node path), addressing any subtree by hash, page-level edit locality (§5), and diffs
that visit a vanishing fraction of nodes (§6). If footprint mattered most, shrink the address
(8-byte short hashes inside pages, full hashes only at page boundaries) — that arithmetic, not
the trie, is the storage lever.

## 4. Geometry: does the digit-path explosion happen?

The classic worry: dense sequential ids (`NCIT_C100000…C139999`) make a naive trie spend one
node per digit. Measured — depth per key, fan-out, node sizes, and the structural bound that
byte-branching (≤ 256 edges) plus one-prefix-per-node puts on node size:

In [4]:
def key_depth(store, root, key):
    node, depth = store.get(root), 1
    while isinstance(node, mrd.Internal):
        key = key[len(node.prefix):]
        if not key:
            return depth
        child = dict(node.edges).get(key[0])
        if child is None:
            return depth
        key, node, depth = key[1:], store.get(child), depth + 1
    return depth

order = mrd.canonical_order(store, root)
internals = [store.get(a) for a in order if isinstance(store.get(a), mrd.Internal)]
fanouts = np.array([len(n.edges) for n in internals])
sizes = np.array([mrd.node_bytes(store, a) for a in set(order)])
sample = random.Random(0).sample(list(ENTRIES), 2000)
depths = np.array([key_depth(store, root, k) for k in sample])

print(f"nodes: {len(set(order)):,} ({len(internals):,} internal)   "
      f"fan-out: mean {fanouts.mean():.1f}, max {fanouts.max()}")
print(f"node size: mean {sizes.mean():.0f} B, max {sizes.max()} B "
      f"(structural bound ≈ 5.7 KB at 256 edges)")
print(f"lookup depth over 2000 sampled keys: mean {depths.mean():.1f}, max {depths.max()}")
print("→ no digit explosion: path compression folds shared runs, and depth is the")
print("  number of BRANCHING decisions on the key, not the key length")

nodes: 55,484 (5,475 internal)   fan-out: mean 10.1, max 11
node size: mean 37 B, max 243 B (structural bound ≈ 5.7 KB at 256 edges)
lookup depth over 2000 sampled keys: mean 8.5, max 9
→ no digit explosion: path compression folds shared runs, and depth is the
  number of BRANCHING decisions on the key, not the key length


## 5. The prolly part: packing nodes into pages

Radix nodes here average well under a disk page, so storage needs to *group* them. Packing the
canonical depth-first node sequence with a content-defined boundary (a node's address masks to
zero, MIN 8 / MAX 128 nodes) yields a page partition that is a pure function of the dictionary —
and edits reshuffle only pages near the changed path:

In [5]:
pages = mrd.pack_pages(store, root)
page_bytes = np.array([sum(mrd.node_bytes(store, a) for a in p) for p in pages])
print(f"{len(pages)} pages; nodes/page mean {np.mean([len(p) for p in pages]):.0f}; "
      f"bytes/page mean {page_bytes.mean():.0f}, p90 {np.percentile(page_bytes, 90):.0f}")

# Edit blast radius at the PAGE level: add 100 new NCIT terms.
root2 = root
for i in range(100):
    root2 = mrd.insert(store, root2, f"http://purl.obolibrary.org/obo/NCIT_C{900000 + i}".encode(), 10**6 + i)
pages2 = mrd.pack_pages(store, root2)
a, b = {tuple(p) for p in pages}, {tuple(p) for p in pages2}
print(f"after adding 100 terms: {len(b - a)} of {len(pages2)} pages new "
      f"({len(a & b)} shared unchanged)")

1395 pages; nodes/page mean 40; bytes/page mean 1453, p90 3015
after adding 100 terms: 6 of 1397 pages new (1391 shared unchanged)


## 6. The diff, pruned by address

Two dictionary versions differing by those 100 added terms. The walk prunes every address-equal
subtree, so it must visit O(changed × depth) nodes, not the ~57k that exist:

In [6]:
changes, visited = mrd.diff(store, root, store, root2)
total = len(set(mrd.canonical_order(store, root2)))
print(f"diff found {len(changes)} changed entries "
      f"(all additions: {all(old is None for old, _ in changes.values())})")
print(f"nodes visited: {visited} of {total:,} — {visited/total:.1%} of the tree")
assert len(changes) == 100 and visited < total // 10

diff found 100 changed entries (all additions: True)
nodes visited: 4 of 55,596 — 0.0% of the tree


## Verdict: could it work as a dictionary implementation?

**Yes — with the design corrected.** What the measurements establish:

| claim from the sketch | verdict |
|---|---|
| history-independent Merkle root | **holds** — but by trie canonicity + sorted serialization, not by any chunker; pinned by hypothesis (any insert order ≡ batch build, byte-identical) |
| prefix dedup slashes footprint | **refuted on this corpus** (§3) — 20-byte addresses cost ≈ what prefix sharing saves (structural ≈ 92% of raw), and flat sorted + zlib is ~10× smaller; the structure is justified by its *operational* properties, not footprint |
| "Gaussian chunker" needed for splits | **refuted** — unnecessary for canonicity; where CDC genuinely enters is page packing (§5) |
| no digit-path explosion on dense ids | **holds** — path compression + byte-branching give mean lookup depth ~8.5 (branching decisions, not key length) and a hard node-size bound (§4) |
| O(changed × depth) diff | **holds** — 100 additions found visiting ~1% of nodes (§6) |
| the sketch's Go PoC proves it | **refuted** — vacuous test (duplicate keys clobber; format string inserts one identical URI) |

What this notebook deliberately does **not** show: concurrency, real disk layout, value
storage for long literals, and the id→string reverse index (a second tree, or ids assigned as
insertion-ordinals into a flat log). Those are engineering, not existence questions — the
existence question is answered by the property tests plus the numbers above.

**Relation to the engine in this repository** *(corrected 2026-07-17 — the first version of
this cell claimed the engine's dictionary is "a flat sorted-key prolly tree (term → ordinal)";
reading the actual source refuted that)*: the engine's dictionary
(`prolly-rdf`'s `prolly-codec` `Dictionary.java`) is a prolly tree keyed by **hash-derived
TermId** mapping id → encoded-term bytes; term → id is *computed* (salted FNV hashing with a
collision-probe), not looked up in a sorted structure at all. That makes the engine's design a
third point in the space: forward mapping ~O(1) by hashing, reverse mapping one Int64 tree
lookup, no prefix structure anywhere. The Java benchmark
(`prolly-rdf4j`'s `dictbench/` in the RDF ring; write-up:
`docs/write-ups/merkle-radix-dictionary-bench.md` there) measured the radix trie against that
real design head-to-head (50k NCIt-shaped terms, 3 forks, Welch-checked): **batch build −46%**
(40.4 vs 74.6 ms — real, t = −30.4), **lookup −66%** heap-form / **−53% through a fully serialized walk**
(0.53 vs 1.13 µs/term — the win survives deserialization, which costs the radix +25.5%;
the engine's flatbuffer framing vs the radix's leaner hand layout remains the one named
residual),
**streaming inserts +132%** (path-copying re-hashes O(depth) nodes per insert — a batching
layer would be mandatory), and serialized footprint **93% vs 140% of raw** (the engine's tree
pays TermId keys + tuple framing on every term; the radix stores shared prefixes once).